```bash
pip install muvit "zarr>=3" matplotlib
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import zarr
from matplotlib.patches import Rectangle
from muvit.data import MuViTDataset
from muvit.mae import MuViTMAE2d

In [ ]:
class MyMuViTDataset(MuViTDataset):
    n_channels = 1  # grayscale EM
    ndim = 2  # 2D crops

    def __init__(self, path, levels=(1, 8, 32), patch_size=256, n_samples=1000, z_range=None, seed=None):
        super().__init__()
        raw = zarr.open_group(f"{path}/raw", mode="r")

        # OME metadata: scale and translation (nm, axes x, y, z) of every pyramid array
        pyramid = {}
        for d in raw.attrs["ome"]["multiscales"][0]["datasets"]:
            t = {t["type"]: t[t["type"]] for t in d["coordinateTransformations"]}
            pyramid[d["path"]] = np.array(t["scale"]), np.array(t.get("translation", [0.0, 0.0, 0.0]))
        s0 = next(iter(pyramid))
        s0_scale = pyramid[s0][0]
        by_factor = {round(scale[0] / s0_scale[0]): p for p, (scale, _) in pyramid.items()}  # XY factor -> array
        if missing := [f for f in levels if f not in by_factor]:
            raise ValueError(f"levels {missing} not in pyramid, available: {sorted(by_factor)}")

        scales, translations = zip(*(pyramid[by_factor[f]] for f in levels))
        self.arrays = [raw[by_factor[f]] for f in levels]  # stored as (x, y, z)
        self.z_factors = [s[2] / s0_scale[2] for s in scales]
        self.pixel_size = [s[[1, 0]] for s in scales]  # (y, x) nm
        self.offset = [t[[1, 0]] for t in translations]  # (y, x) nm

        self._levels = tuple(levels)
        self.patch_size = patch_size
        self.n_samples = n_samples
        self.shape0 = raw[s0].shape
        self.z_range = z_range or (0, self.shape0[2])
        if patch_size * max(levels) > min(self.shape0[:2]):
            raise ValueError(f"patch_size * max(levels) = {patch_size * max(levels)} exceeds the XY size {self.shape0[:2]}")
        self.seed = seed  # None: new random crops every epoch; int: fixed crops (validation)

    @property
    def levels(self):
        return self._levels

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        rng = np.random.default_rng(None if self.seed is None else self.seed + idx)
        P = self.patch_size
        # random centre (s0 pixels, x and y) where the coarsest crop fits, and a random z slice
        half = P * max(self.levels) / 2
        center = [rng.uniform(half, n - half) for n in self.shape0[:2]]
        z = rng.integers(*self.z_range)

        imgs, bbox = [], []
        for f, arr, zf, px, off in zip(self.levels, self.arrays, self.z_factors, self.pixel_size, self.offset):
            x0, y0 = (int(np.clip(round(c / f - P / 2), 0, n - P)) for c, n in zip(center, arr.shape[:2]))
            zi = min(int(z // zf), arr.shape[2] - 1)
            imgs.append(arr[x0:x0 + P, y0:y0 + P, zi].T)  # (x, y) -> (y, x)
            first = np.array([y0, x0])
            bbox.append([first * px + off, (first + P - 1) * px + off])

        return {
            "img": torch.from_numpy(np.stack(imgs)[:, None]).float() / 255,  # (L, C, Y, X)
            "bbox": torch.tensor(np.array(bbox), dtype=torch.float32),  # (L, 2, 2) in nm
        }

In [ ]:
path = "/groups/miaai/miaai/lmd-v0.0.1/data/em-drosophila-FANC-vnc/crop-001.zarr"
levels = (1, 2, 4)

train_ds = MyMuViTDataset(path, levels=levels, n_samples=4000, z_range=(0, 1320))
val_ds = MyMuViTDataset(path, levels=levels, n_samples=200, z_range=(1320, 1466), seed=0)

train_dl = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl = torch.utils.data.DataLoader(val_ds, batch_size=16, shuffle=False)

In [ ]:
batch = next(iter(train_dl))
img, bbox = batch["img"], batch["bbox"]  # (B, L, C, Y, X), (B, L, 2, 2)
print(bbox.shape, img.shape)
print(bbox)

n_rows = 4
fig, axes = plt.subplots(n_rows, len(levels), figsize=(3 * len(levels), 3 * n_rows), squeeze=False)
for b in range(n_rows):
    for l, f in enumerate(levels):
        ax = axes[b, l]
        ax.imshow(img[b, l, 0], cmap="gray", vmin=0, vmax=1)
        ax.axis("off")
        if b == 0:
            ax.set_title(f"level {f}")
        if l > 0:  # finer crop's bbox, from nm to this level's pixels
            (y0, x0), (y1, x1) = ((bbox[b, l - 1] - bbox[b, l, 0]) / torch.from_numpy(train_ds.pixel_size[l])).tolist()
            ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color="r"))
plt.tight_layout()

In [ ]:
model = MuViTMAE2d(
    in_channels=train_ds.n_channels,
    levels=train_ds.levels,
    patch_size=8,
    num_layers=12,
    num_layers_decoder=4,
)

model.fit(train_dl, val_dl, output="/path/to/pretrained", num_epochs=5)